# ST-04 — Siretisation, phase 3 : matching approfondi

Reprend les etablissements non resolus en phase 2 et ajoute le code APE et
l'annee d'ouverture au scoring.

Cote SIRENE, la comparaison se fait sur `activitePrincipaleEtablissement` et
`dateCreationEtablissement`, au niveau etablissement donc, et non au niveau de
l'unite legale.

Cote FINESS, les deux informations viennent des extractions
`Extraction_EG_codeape*.csv et Extraction_EG_dateouverture*.csv` deposees dans
`data/referentiels/`.

Comme en phase 2, les candidats proviennent de l'echantillon filtre de ST-02bis.


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
from src.siretisation import CONFIG_ST, COLS_ETAB_EXPORT
from src.phases import executer_phase
from src import domaine_ape
from src.scoring import extraire_annee
from src.excel_export import export_topn_excel, LABELS
from src.display import afficher_tableau, afficher_synthese
from config.settings import (
    ST_PERIMETRE, SIRENE_ETAB_FILTRE, SIRENE_ETAB_FILTRE_PARTITIONNE,
    ST_VALIDES_P1, ST_PHASE2, ST_PHASE3, RESULTS_ST_DIR, BUDGET_MEMOIRE_GO,
)

RESULTS_ST_DIR.mkdir(parents=True, exist_ok=True)
NON_RESOLUS = {"DOUTEUX", "REJETE", "SANS_CANDIDAT", "NON_FIABLE_APE"}

sirets_valides_p1 = set(
    pd.read_parquet(ST_VALIDES_P1)["siret_norm_ege"].dropna().astype(str))
sirets_valides_p1.discard("")

In [2]:
df_p2 = pd.read_excel(ST_PHASE2, sheet_name="Top5", dtype=str)
df_p2["rang"] = pd.to_numeric(df_p2["rang"], errors="coerce")
rang1 = df_p2[df_p2["rang"] == 1]
COL_STATUT = "statut_structure" if "statut_structure" in rang1.columns else "statut_candidat"
ids_phase3 = set(rang1[rang1[COL_STATUT].isin(NON_RESOLUS)]["NB_EgeId"]
                 .dropna().astype(str))

df_perimetre = pd.read_parquet(ST_PERIMETRE)
df_phase3 = df_perimetre[
    df_perimetre["NB_EgeId"].astype(str).isin(ids_phase3)
].copy().reset_index(drop=True)
print(f"EGE a traiter en phase 3 : {len(df_phase3):,}")

EGE a traiter en phase 3 : 11,936


In [3]:
from src.referentiel import diagnostiquer
display(diagnostiquer())

,niveau,type,fichier,lignes,renseignes,colonnes,exemples
0,pm,ape,Extraction_PM_codeape.csv,98158,31170,nofiness | codeape,010000016=8710A | 010000156=8610Z | 010000164=...
1,pm,date,Extraction_PM_datecreat.csv,98158,98158,nofiness | dateouvert,010000016=01/01/1994 | 010000156=01/01/2001 | ...
2,ege,ape,Extraction_EG_codeape.csv,174438,67292,nofiness | codeape,010000024=8610Z | 010000032=8610Z | 010000040=...
3,ege,date,Extraction_EG_dateouverture_20260728.csv,174438,170225,nofiness | dateouvert,010000024=1979-02-13 | 010000032=1901-01-01 | ...


In [4]:
# APE et annee d'ouverture sont deja portes par le perimetre, enrichi en ST-02.
print(f"APE renseigne   : {(df_phase3['ape_referentiel'] != '').sum():,} / {len(df_phase3):,}")
print(f"Annee renseignee: {df_phase3['annee_referentiel'].notna().sum():,} / {len(df_phase3):,}")


APE renseigne   : 4,760 / 11,936
Annee renseignee: 11,428 / 11,936


In [5]:
def preparer_bloc_etab(df):
    """Annee de creation de l'etablissement, derivee bloc par bloc."""
    df = df.copy()
    if "dateCreationEtablissement" in df.columns:
        df["annee_creation_etab"] = df["dateCreationEtablissement"].apply(extraire_annee)
    return df

df_top3 = executer_phase(
    df_phase3, CONFIG_ST,
    chemin_partitionne=SIRENE_ETAB_FILTRE_PARTITIONNE,
    chemin_complet=SIRENE_ETAB_FILTRE,
    n=3, approfondi=True,
    identifiants_deja_valides=sirets_valides_p1,
    colonnes_sirene=COLS_ETAB_EXPORT + ["annee_creation_etab"],
    budget_go=BUDGET_MEMOIRE_GO,
    preparer_bloc=preparer_bloc_etab,
    niveau_matching="departement",
)

df_top3 = df_top3.rename(columns={"siret": "siret_ref_app",
                                  "nom_sirene_retenu": "nom_etab_retenu",
                                  "identifiant_coherent": "siret_coherent",
                                  "score_ajuste": "score_approfondi_ajuste"})

rang1 = df_top3[df_top3["rang"] == 1]
print(f"\nLignes top 3 : {len(df_top3):,}")
print(f"   avec score APE  : {df_top3['score_ape'].notna().sum():,}")
print(f"   avec score date : {df_top3['score_date'].notna().sum():,}")
print()
print(rang1["statut_candidat"].value_counts())

1 structures sans departement exploitable


Phase 3 (departement):   0%|          | 0/104 [00:00<?, ?it/s]

/home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/src/phases.py:139: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(resultats + orphelins, ignore_index=True) if (resultats or orphelins) \



Lignes top 3 : 35,804
   avec score APE  : 14,280
   avec score date : 34,282

statut_candidat
DOUTEUX          8547
VALIDE           2700
REJETE            617
VALIDE_FORT        71
SANS_CANDIDAT       1
Name: count, dtype: int64


## Contrôle du domaine d'activité

Une validation par le score ne suffit pas. Un candidat peut avoir le bon nom et
la bonne adresse mais une activité étrangère au champ FINESS — une SCI portant
les murs d'un établissement, par exemple.

Un candidat n'est retenu que si son code APE est identique à celui de la
structure, ou s'il relève du domaine sanitaire, social ou médico-social.
L'égalité exacte prime. Parmi les candidats validés par le score, on retient le
mieux classé qui satisfait cette condition.

Quand aucun candidat validé n'est acceptable, la structure ressort en
`NON_FIABLE_APE` : le score la validait, l'activité l'en empêche.

In [6]:
df_top3 = domaine_ape.appliquer_controle(
    df_top3, col_id="NB_EgeId",
    col_ape_candidat="activitePrincipaleEtablissement",
    col_ape_structure="ape_referentiel",
)

rang1 = df_top3[df_top3["rang"] == 1]

afficher_tableau(domaine_ape.rapport(df_top3, "NB_EgeId"),
                 "Effet du controle d'activite", max_lignes=12)
print()
print(rang1["statut_structure"].value_counts().to_string())


Indicateur,Nombre
Structures traitées,11936
Validées par le score au rang 1,2771
—,
Validées après contrôle d'activité,1742
dont candidat de rang 1 conservé,1610
dont candidat remplacé,132
Bloquées — NON_FIABLE_APE,1578
—,
retenues sur APE identique,1014
retenues sur le domaine,728



statut_structure
DOUTEUX           8007
VALIDE            1741
NON_FIABLE_APE    1578
REJETE             608
VALIDE_FORT          1
SANS_CANDIDAT        1


In [7]:
afficher_tableau(
    df_top3[["NB_EgeId", "TX_NomEgeLong", "siret_ref_app",
             "denominationUniteLegale", "nom_etab_retenu",
             "ape_finess", "activitePrincipaleEtablissement", "ape_acceptable",
             "candidat_retenu", "score_ape",
             "annee_referentiel", "annee_creation_etab", "score_date",
             "score_global", "score_global_approfondi",
             "statut_candidat", "statut_structure", "rang"]],
    "Apercu top 3 phase 3", max_lignes=10)

NB_EgeId,TX_NomEgeLong,siret_ref_app,denominationUniteLegale,nom_etab_retenu,ape_finess,activitePrincipaleEtablissement,ape_acceptable,candidat_retenu,score_ape,annee_referentiel,annee_creation_etab,score_date,score_global,score_global_approfondi,statut_candidat,statut_structure,rang
14,CENTRE HOSPITALIER PUBLIC DE HAUTEVILLE - UNITE ESPERANCE,26011019200044,CTRE HOSPITALIER PUBLIC D HAUTEVILLE,CTRE MEDICAL ESPERANCE,8610Z,86.10Z,True,False,100.000000,1904.000000,2000.000000,0.000000,42.890000,50.030000,DOUTEUX,DOUTEUX,1
14,CENTRE HOSPITALIER PUBLIC DE HAUTEVILLE - UNITE ESPERANCE,26011019200010,CTRE HOSPITALIER PUBLIC D HAUTEVILLE,CTRE HOSPITALIER PUBLIC HAUTEVILLE,8610Z,86.10Z,True,False,100.000000,1904.000000,2000.000000,0.000000,50.660000,55.460000,DOUTEUX,,2
14,CENTRE HOSPITALIER PUBLIC DE HAUTEVILLE - UNITE ESPERANCE,77554456200017,ORSAC,CENTRE ORCET MANGINI,8610Z,86.10Z,True,False,100.000000,1904.000000,1900.000000,60.000000,39.520000,53.660000,DOUTEUX,,3
16,CENTRE HOSPITALIER PUBLIC DE HAUTEVILLE - UNITE ALBARINE,26011019200028,CTRE HOSPITALIER PUBLIC D HAUTEVILLE,UNITE ALBARINE,8610Z,86.10Z,True,False,100.000000,1974.000000,2000.000000,0.000000,39.770000,47.840000,DOUTEUX,DOUTEUX,1
16,CENTRE HOSPITALIER PUBLIC DE HAUTEVILLE - UNITE ALBARINE,26011019200010,CTRE HOSPITALIER PUBLIC D HAUTEVILLE,CTRE HOSPITALIER PUBLIC HAUTEVILLE,8610Z,86.10Z,True,False,100.000000,1974.000000,2000.000000,0.000000,50.800000,55.560000,DOUTEUX,,2
16,CENTRE HOSPITALIER PUBLIC DE HAUTEVILLE - UNITE ALBARINE,77554456200017,ORSAC,CENTRE ORCET MANGINI,8610Z,86.10Z,True,False,100.000000,1974.000000,1900.000000,0.000000,40.010000,48.000000,DOUTEUX,,3
21,MARPA ANNEXE DE LA MARPA DE MANZIAT,22010001000143,DEPARTEMENT DE L AIN,CENTRE MEDICO SOCIAL,,88.99B,True,False,nan,1995.000000,1994.000000,100.000000,32.950000,46.360000,DOUTEUX,DOUTEUX,1
21,MARPA ANNEXE DE LA MARPA DE MANZIAT,40499931000018,ASS LA GRANGE AUX LOUPS,ASS GRANGE LOUPS,,87.90A,True,False,nan,1995.000000,1996.000000,100.000000,27.370000,41.900000,DOUTEUX,,2
21,MARPA ANNEXE DE LA MARPA DE MANZIAT,52469786900013,ASSOCIATION SPORTIVE DU COLLEGE DE BAGE-LA-VILLE,SPORTIVE COLLEGE BAGE VILLE,,93.12Z,False,False,nan,1995.000000,1983.000000,20.000000,46.350000,41.080000,DOUTEUX,,3
27,MARPA LA VALETTE,40311970400019,ASS GESTION MARPA DU SECTEUR MONTRACOL,ASS GESTION MARPA SECTEUR MONTRACOL,8730A,87.30A,True,False,100.000000,1995.000000,1995.000000,100.000000,29.570000,50.700000,DOUTEUX,DOUTEUX,1


In [8]:
COLS_EXPORT = [
    "ID_Ege", "NB_EgeId", "NB_PmSmsseId", "TX_NumFinessEge", "TX_Siret",
    "TX_CategorieEntiteGeographiqueExercice", "TX_NomEgeLong",
    "cdcommune_norm_ege", "adresse_complete_ege",
    "ape_finess", "annee_referentiel",

    "siret_ref_app", "siren", "siret_coherent", "bonus_applique",
    "denominationUniteLegale", "enseigne1Etablissement",
    "denominationUsuelleEtablissement", "nom_etab_retenu",
    "adresse_complete_etab", "codeCommuneEtablissement",
    "categorieJuridiqueUniteLegale", "activitePrincipaleEtablissement",
    "dateCreationEtablissement", "annee_creation_etab",

    "ape_candidat", "ape_identique", "ape_dans_domaine", "ape_acceptable",
    "candidat_retenu", "rang_retenu", "motif_ape",

    "score_nom", "score_adresse", "score_global",
    "score_ape", "score_date", "score_global_approfondi",
    "score_approfondi_ajuste",
    "statut_candidat", "statut_structure", "rang",
]

df_top3["rang"] = df_top3["rang"].astype(int)
compteurs = export_topn_excel(df_top3, ST_PHASE3, COLS_EXPORT, sheet_name="Top3",
                              col_statut="statut_structure")

afficher_synthese({LABELS[s]: n for s, n in compteurs.items()},
                  "Synthese phase 3 siretisation (rang 1)")
print(f"\nFichier : {ST_PHASE3}")


Statut,Nb,% du total
Valide_fort,1,0.0%
Valide,"1,741",14.6%
Douteux,"8,007",67.1%
Rejeté,608,5.1%
Sans_candidat,1,0.0%
Non_fiable_APE,"1,578",13.2%
TOTAL,"11,936",100.0%



Fichier : /home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/results/siretisation/siretisation_phase3_approfondi.xlsx
